# Market Data Quality Analytics for VaR

**Aby Joe Jose** | [GitHub](https://github.com/AbyJoeJose) | [Portfolio](https://abyjoejose.github.io/Data-Science-Portfolio) | abyjoejose00@gmail.com

## What this project does

Value at Risk is computed from historical market data. If that data is wrong, the risk number is wrong, and the firm holds the wrong amount of capital against its trading book.

This notebook builds a small framework that scores market data quality, detects outliers, imputes missing values, and then measures **how much the defects actually move VaR**. That last step is the point. Everything before it is machinery.

The question it answers: *if nobody cleaned this data, how wrong would the risk number be?*

## Why the answer is not obvious

Different defects damage different parts of the calculation, and they do not all push the same way.

* A **bad print** (fat finger, decimal slip, vendor error) creates a return larger than anything real in the sample. It lands in the tail and lifts the variance, so it pushes **both** kinds of VaR up.
* A **stale feed** repeats the previous price, producing runs of zero returns. Those zeros dilute measured volatility and pull **variance based** VaR down, while leaving the extreme days untouched, so the empirical tail barely moves.
* A **gap that gets forward filled** looks similar but behaves differently. The filled day is a zero return, but the *next* day's return then spans two days. In a sample containing March 2020, a merged two day move can exceed anything real and becomes the tail, so **quantile based** VaR goes up while variance hardly changes.

So the honest answer is that it depends on which model you run. Variance based measures are damaged by dilution; quantile based measures are damaged by merging. Section 7 measures all three separately and the numbers come out in opposite directions, which is the reason the notebook reports two models side by side rather than one.

## How it is built

Three small class hierarchies, each with an abstract base and concrete implementations:

| Hierarchy | Implementations |
| --- | --- |
| `OutlierDetector` | z-score, modified z-score (MAD), rolling z-score, EWMA volatility scaled |
| `Imputer` | forward fill, linear interpolation, peer scaled |
| `VaRModel` | historical simulation, parametric, Monte Carlo |

Adding a new method means writing one subclass. Nothing else changes. That is the part that makes it a framework rather than a script.

## Contents

1. Setup and data
2. Quality scorecard
3. Outlier detectors
4. Detector benchmark
5. Imputation and its benchmark
6. VaR models
7. The impact of dirty data on VaR
8. VaR backtesting
9. Vectorisation
10. Limitations and future work


## 1. Setup and data

Six series across five asset classes, all from FRED. Multiple asset classes matter here because each has its own holiday calendar, so the missing data is genuine rather than invented.

| Series | Asset class | Note |
| --- | --- | --- |
| `SP500` | Equity | S&P 500 index level |
| `DEXUSEU` | FX | USD per EUR |
| `DEXJPUS` | FX | JPY per USD |
| `DCOILWTICO` | Commodity | WTI crude, spot |
| `DGS10` | Rates | 10y Treasury yield |
| `VIXCLS` | Volatility | VIX index |


In [1]:
import os
import glob
import time
from abc import ABC, abstractmethod

import numpy as np
import pandas as pd
from scipy import stats

import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

pd.set_option("display.width", 150)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

# config
data_dir = "data"
start_date = "2016-01-01"
confidence = 0.99          # VaR confidence level
portfolio_value = 10_000_000
os.makedirs(data_dir, exist_ok=True)

# chart palette, checked for colourblind separation
blue, orange, green, purple, gold = "#2E6FD9", "#E07B39", "#159C7E", "#A855C7", "#C9A227"
ink, muted, grid = "#1F2933", "#6B7280", "#E5E7EB"
palette = [blue, orange, green, purple, gold]

pio.templates["clean"] = go.layout.Template(layout=dict(
    font=dict(family="Helvetica, Arial, sans-serif", size=13, color=ink),
    title=dict(font=dict(size=16, color=ink), x=0.0, xanchor="left"),
    paper_bgcolor="white", plot_bgcolor="white", colorway=palette,
    xaxis=dict(showgrid=True, gridcolor=grid, zeroline=False, linecolor=grid),
    yaxis=dict(showgrid=True, gridcolor=grid, zeroline=False, linecolor=grid),
    legend=dict(orientation="h", yanchor="bottom", y=1.0, x=0,
                bgcolor="rgba(0,0,0,0)", font=dict(size=11)),
    margin=dict(l=70, r=30, t=70, b=55),
))
pio.templates.default = "clean"

print("setup complete")
print(f"pandas {pd.__version__}  numpy {np.__version__}")

setup complete
pandas 3.0.3  numpy 2.4.4


### Loading the data

FRED's CSV endpoint did not respond to Python from the machine this was built on, so the six series were downloaded through a browser and are read from `data/`. Reading from disk also makes the notebook reproducible: the cached panel pins the exact data behind every number below.

To refresh, download each series and drop the file in `data/`:

```
https://fred.stlouisfed.org/graph/fredgraph.csv?id=SP500&cosd=2016-01-01
https://fred.stlouisfed.org/graph/fredgraph.csv?id=DEXUSEU&cosd=2016-01-01
https://fred.stlouisfed.org/graph/fredgraph.csv?id=DEXJPUS&cosd=2016-01-01
https://fred.stlouisfed.org/graph/fredgraph.csv?id=DCOILWTICO&cosd=2016-01-01
https://fred.stlouisfed.org/graph/fredgraph.csv?id=DGS10&cosd=2016-01-01
https://fred.stlouisfed.org/graph/fredgraph.csv?id=VIXCLS&cosd=2016-01-01
```


In [2]:
series_map = {
    "SP500":      "sp500",
    "DEXUSEU":    "eurusd",
    "DEXJPUS":    "usdjpy",
    "DCOILWTICO": "wti",
    "DGS10":      "ust10",
    "VIXCLS":     "vix",
}

# how each series turns into a return
# log: price like, use log differences
# diff: already a rate, use absolute daily change
return_kind = {"sp500": "log", "eurusd": "log", "usdjpy": "log",
               "wti": "log", "ust10": "diff", "vix": "log"}

# the four price series that make up the traded portfolio
portfolio_assets = ["sp500", "eurusd", "usdjpy", "wti"]

cache_path = os.path.join(data_dir, "market_panel.csv")


def read_fred_csv(path):
    """Read one FRED csv into a date indexed numeric frame."""
    df = pd.read_csv(path, na_values=[".", "", "NA"])
    date_col = next(c for c in df.columns
                    if c.strip().lower() in ("date", "observation_date"))
    df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
    df = df.dropna(subset=[date_col]).set_index(date_col)
    df.index.name = "date"
    return df.apply(pd.to_numeric, errors="coerce")


def load_panel():
    """Load the cached panel, or build it from the downloaded csv files."""
    if os.path.exists(cache_path):
        panel = pd.read_csv(cache_path, index_col=0, parse_dates=True)
        print(f"loaded cache: {cache_path}  ({len(panel):,} rows)")
        return panel

    files = sorted(f for f in glob.glob(os.path.join(data_dir, "*.csv"))
                   if os.path.abspath(f) != os.path.abspath(cache_path))
    if not files:
        raise FileNotFoundError(
            f"no source csv files in {data_dir}/. download these six from FRED:\n  "
            + "\n  ".join(series_map))

    frames = []
    for f in files:
        df = read_fred_csv(f)
        frames.append(df)
        print(f"  {os.path.basename(f):<20} {len(df):>6,} rows  {', '.join(df.columns)}")

    panel = pd.concat(frames, axis=1).sort_index()
    panel = panel.loc[:, ~panel.columns.duplicated()]

    missing = [s for s in series_map if s not in panel.columns]
    if missing:
        raise ValueError(f"missing series: {', '.join(missing)}. "
                         f"found: {', '.join(panel.columns)}")

    panel.to_csv(cache_path, encoding="utf-8")
    print(f"\nsaved cache to {cache_path}")
    return panel


raw = load_panel()

# keep only the six we asked for, rename, and put on a business day grid
raw = raw[[c for c in series_map if c in raw.columns]].rename(columns=series_map)
raw = raw.loc[raw.index >= start_date]
raw = raw.reindex(pd.bdate_range(raw.index.min(), raw.index.max()))
raw.index.name = "date"

print(f"\npanel: {raw.shape[0]:,} business days x {raw.shape[1]} series")
print(f"range: {raw.index.min().date()} to {raw.index.max().date()}")
raw.tail(3)

loaded cache: data\market_panel.csv  (2,799 rows)

panel: 2,799 business days x 6 series
range: 2016-01-04 to 2026-09-24


,sp500,eurusd,usdjpy,wti,ust10,vix
date,,,,,,
2026-09-22,"7,764.6400",NaN,NaN,96.4100,4.9600,14.2100
2026-09-23,"7,706.0300",NaN,NaN,NaN,5.1100,NaN
2026-09-24,"7,704.1300",NaN,NaN,NaN,NaN,NaN


## 2. Quality scorecard

Before any risk number gets computed, the inputs need grading. `DataQualityReport` scores each series on five checks and rolls them into a letter grade.

| Check | What it catches | Why it matters for VaR |
| --- | --- | --- |
| Completeness | Missing observations | Gaps have to be filled somehow, and the choice changes volatility |
| Staleness | Runs of identical prices | Repeated prices produce zero returns and understate risk |
| Max gap | The longest unbroken run of missing data | A long gap is a different problem from scattered holidays |
| Jump rate | Share of daily moves beyond 5 sigma | Flags both bad prints and genuine stress |
| Non positive | Prices at or below zero | Breaks log returns outright |

The last check exists because of a real event. WTI crude settled at **negative $37** on 20 April 2020. Any pipeline that takes log returns of a price series without checking the sign produces `NaN` that day and, if it silently drops them, quietly loses the largest move in the sample.


In [3]:
class DataQualityReport:
    """Scores a panel of market data time series on five quality checks."""

    def __init__(self, panel, stale_run=2, jump_sigma=5.0):
        self.panel = panel
        self.stale_run = stale_run      # a repeat this long counts as stale
        self.jump_sigma = jump_sigma

    def completeness(self, s):
        """Share of business days with an observation, within the live range."""
        live = s.loc[s.first_valid_index():s.last_valid_index()]
        return float(live.notna().mean() * 100)

    def staleness(self, s):
        """Share of days whose value repeats the previous one."""
        live = s.dropna()
        if len(live) < 2:
            return 0.0
        return float((live.diff() == 0).mean() * 100)

    def max_gap(self, s):
        """Longest run of consecutive missing business days."""
        live = s.loc[s.first_valid_index():s.last_valid_index()]
        miss = live.isna().astype(int)
        if miss.sum() == 0:
            return 0
        groups = (miss.diff() != 0).cumsum()
        runs = miss.groupby(groups).sum()
        return int(runs.max())

    def jump_rate(self, s):
        """Share of daily moves larger than jump_sigma full sample sigmas."""
        r = s.dropna().pct_change().dropna()
        if len(r) < 30 or r.std() == 0:
            return 0.0
        z = (r - r.mean()) / r.std()
        return float((z.abs() > self.jump_sigma).mean() * 100)

    def non_positive(self, s):
        """Count of observations at or below zero. Breaks log returns."""
        return int((s.dropna() <= 0).sum())

    @staticmethod
    def _grade(completeness, staleness, jumps):
        """Roll the checks into a letter grade. Deliberately simple and tunable."""
        score = 100.0
        score -= (100 - completeness) * 2.0     # missing data is expensive
        score -= staleness * 1.5                # staleness understates risk
        score -= jumps * 10.0                   # jumps are rare, so weight heavily
        score = max(score, 0.0)
        for cutoff, letter in [(90, "A"), (80, "B"), (70, "C"), (60, "D")]:
            if score >= cutoff:
                return letter, round(score, 1)
        return "F", round(score, 1)

    def summary(self):
        """One row per series with every check and the resulting grade."""
        rows = []
        for col in self.panel.columns:
            s = self.panel[col]
            comp = self.completeness(s)
            stale = self.staleness(s)
            jump = self.jump_rate(s)
            letter, score = self._grade(comp, stale, jump)
            rows.append({
                "series": col,
                "first": s.first_valid_index().date(),
                "last": s.last_valid_index().date(),
                "obs": int(s.notna().sum()),
                "completeness_pct": round(comp, 2),
                "stale_pct": round(stale, 2),
                "max_gap_days": self.max_gap(s),
                "jump_rate_pct": round(jump, 3),
                "non_positive": self.non_positive(s),
                "score": score,
                "grade": letter,
            })
        return pd.DataFrame(rows).set_index("series")


report = DataQualityReport(raw)
scorecard = report.summary()
print("market data quality scorecard\n")
print(scorecard.to_string())

market data quality scorecard

             first        last   obs  completeness_pct  stale_pct  max_gap_days  jump_rate_pct  non_positive   score grade
series                                                                                                                    
sp500   2016-09-26  2026-09-24  2513           96.3200     0.0400             1         0.4380             0 88.2000     B
eurusd  2016-01-04  2026-09-18  2677           95.7800     0.9300             2         0.0370             0 89.8000     B
usdjpy  2016-01-04  2026-09-18  2677           95.7800     1.0100             2         0.1120             0 88.9000     B
wti     2016-01-04  2026-09-22  2682           95.8900     0.6000             2         0.1490             1 89.4000     B
ust10   2016-01-04  2026-09-23  2682           95.8500     8.6100             1         0.4100             0 74.7000     C
vix     2016-01-04  2026-09-22  2728           97.5300     0.4400             1         0.4770             0

In [4]:
# flag anything that needs attention before it reaches the risk engine
print("findings\n")

for name, row in scorecard.iterrows():
    notes = []
    if row["completeness_pct"] < 97:
        notes.append(f"{100 - row['completeness_pct']:.1f}% missing")
    if row["stale_pct"] > 5:
        notes.append(f"{row['stale_pct']:.1f}% stale")
    if row["max_gap_days"] > 3:
        notes.append(f"{row['max_gap_days']}d gap")
    if row["non_positive"] > 0:
        notes.append(f"{row['non_positive']} non positive values")
    if notes:
        print(f"  {name:<8} grade {row['grade']}  {', '.join(notes)}")

# the negative price case, called out explicitly
neg = raw[portfolio_assets].le(0).any()
if neg.any():
    for col in neg[neg].index:
        bad = raw.loc[raw[col] <= 0, col]
        print(f"\n  {col} went non positive on "
              f"{', '.join(str(d.date()) for d in bad.index[:5])}"
              f"{' and others' if len(bad) > 5 else ''}, low of {bad.min():.2f}")
        print(f"  log returns are undefined here, so these days are masked "
              f"rather than silently dropped.")
else:
    print("\n  no non positive prices in the sample")

findings

  sp500    grade B  3.7% missing
  eurusd   grade B  4.2% missing
  usdjpy   grade B  4.2% missing
  wti      grade B  4.1% missing, 1 non positive values
  ust10    grade C  4.2% missing, 8.6% stale

  wti went non positive on 2020-04-20, low of -36.98
  log returns are undefined here, so these days are masked rather than silently dropped.


In [5]:
fig = make_subplots(rows=2, cols=1, shared_xaxes=False, vertical_spacing=0.18,
                    subplot_titles=("Quality score by series",
                                    "Completeness and staleness (%)"))

colors = [green if g == "A" else gold if g in ("B", "C") else orange
          for g in scorecard["grade"]]
fig.add_trace(go.Bar(x=scorecard.index, y=scorecard["score"], marker_color=colors,
                     text=[f"{s:.0f} ({g})" for s, g in
                           zip(scorecard["score"], scorecard["grade"])],
                     textposition="outside", showlegend=False,
                     hovertemplate="%{x}: score %{y:.1f}<extra></extra>"),
              row=1, col=1)

fig.add_trace(go.Bar(x=scorecard.index, y=scorecard["completeness_pct"],
                     name="complete", marker_color=blue,
                     hovertemplate="%{x}: %{y:.1f}% complete<extra></extra>"),
              row=2, col=1)
fig.add_trace(go.Bar(x=scorecard.index, y=scorecard["stale_pct"],
                     name="stale", marker_color=orange,
                     hovertemplate="%{x}: %{y:.1f}% stale<extra></extra>"),
              row=2, col=1)

fig.update_yaxes(title="score", row=1, col=1, range=[0, 115])
fig.update_yaxes(title="%", row=2, col=1)
fig.update_layout(height=640, barmode="group",
                  title="Exhibit 1. Not every series arrives in the same condition")
fig.show()

### Turning prices into returns

Two details that are easy to get wrong.

**Yields are not prices.** `DGS10` is already a rate, so the return is the absolute daily change in percentage points, not a percentage change. A move from 0.5% to 0.6% is 10bp, not a 20% return. Treating it as a percentage change would make the 2020 rates data look far more volatile than it was.

**Log returns need positive prices.** The `np.where` guard below masks any day where either the current or previous price is at or below zero. Without it, negative WTI in April 2020 silently poisons the series.


In [6]:
def compute_returns(panel, kinds):
    """Convert a price panel into returns, respecting each series' type."""
    out = {}
    for col, kind in kinds.items():
        if col not in panel.columns:
            continue
        s = panel[col]
        if kind == "log":
            prev = s.shift(1)
            valid = (s > 0) & (prev > 0)
            out[col] = pd.Series(
                np.where(valid, np.log(s.where(valid) / prev.where(valid)), np.nan),
                index=s.index)
        else:
            out[col] = s.diff()
    return pd.DataFrame(out)


def build_portfolio(ret_frame, assets=None, min_assets=3):
    """Equal weighted portfolio return. A day needs at least min_assets quotes.

    The same rule is applied to every version of the data (clean, dirty,
    repaired) so the comparisons later on are like for like. Without a fixed
    rule the clean panel would be measured over fewer days than the dirty one,
    and the difference would partly be sample size rather than data quality.
    """
    assets = assets or portfolio_assets
    sub = ret_frame[assets]
    port = sub.mean(axis=1, skipna=True)
    return port.where(sub.notna().sum(axis=1) >= min_assets).dropna()


returns = compute_returns(raw, return_kind)
portfolio = build_portfolio(returns)

print("annualised volatility\n")
for col in returns.columns:
    r = returns[col].dropna()
    if return_kind[col] == "log":
        # log returns are a fraction, so scale to percent
        vol, unit = r.std() * np.sqrt(252) * 100, "%"
    else:
        # already in percentage points, so no rescaling
        vol, unit = r.std() * np.sqrt(252), " pp"
    print(f"  {col:<8} {vol:7.2f}{unit}   ({len(r):,} observations)")

print(f"\nportfolio: equal weighted across {', '.join(portfolio_assets)}")
print(f"  observations   {len(portfolio):,}")
print(f"  ann volatility {portfolio.std() * np.sqrt(252) * 100:.2f}%")
print(f"  worst day      {portfolio.min() * 100:.2f}% on "
      f"{portfolio.idxmin().date()}")
print(f"  best day       {portfolio.max() * 100:.2f}% on "
      f"{portfolio.idxmax().date()}")

annualised volatility

  sp500      18.16%   (2,416 observations)
  eurusd      7.13%   (2,561 observations)
  usdjpy      9.26%   (2,561 observations)
  wti        50.10%   (2,571 observations)
  ust10       0.83 pp   (2,565 observations)
  vix       124.91%   (2,658 observations)

portfolio: equal weighted across sp500, eurusd, usdjpy, wti
  observations   2,537
  ann volatility 14.68%
  worst day      -9.46% on 2020-03-09
  best day       11.18% on 2020-04-22


## 3. Outlier detectors

Four methods, one base class. `OutlierDetector` defines the interface and the `detect` logic; each subclass only has to say how it scores a point. Adding a fifth method means writing a `score` method and nothing else.

| Method | Scores against | Weakness |
| --- | --- | --- |
| `ZScore` | Full sample mean and standard deviation | The outliers inflate the same standard deviation used to find them |
| `ModifiedZScore` | Median and MAD | Robust to contamination, but still a single full sample estimate |
| `RollingZScore` | Trailing window mean and standard deviation | Adapts to changing volatility, but the window has to be chosen |
| `EwmaVol` | Exponentially weighted volatility | Reacts faster to volatility shifts |

The rolling and EWMA versions both call `.shift(1)` on their estimates. That is deliberate: without it, the day being tested contributes to the volatility it is judged against, which is look ahead bias and makes large moves look normal.


In [7]:
class OutlierDetector(ABC):
    """Base class for outlier detection on a return series."""

    def __init__(self, threshold=4.0):
        self.threshold = threshold

    @property
    def name(self):
        return self.__class__.__name__.replace("Detector", "")

    @abstractmethod
    def score(self, s):
        """Return a signed score per observation. Large absolute value is unusual."""
        ...

    def detect(self, s):
        """Boolean mask of observations beyond the threshold."""
        return self.score(s).abs() > self.threshold

    def __repr__(self):
        return f"{self.name}(threshold={self.threshold})"


class ZScoreDetector(OutlierDetector):
    """Classic z-score against the full sample."""

    def score(self, s):
        sd = s.std(ddof=0)
        if sd == 0:
            return pd.Series(0.0, index=s.index)
        return (s - s.mean()) / sd


class ModifiedZScoreDetector(OutlierDetector):
    """Median and MAD based. The 0.6745 factor makes MAD comparable to sigma."""

    def score(self, s):
        med = s.median()
        mad = (s - med).abs().median()
        if mad == 0:
            mad = s.std(ddof=0) * 0.6745
        if mad == 0 or np.isnan(mad):
            return pd.Series(0.0, index=s.index)
        return 0.6745 * (s - med) / mad


class RollingZScoreDetector(OutlierDetector):
    """Z-score against a trailing window, so the bar moves with volatility."""

    def __init__(self, threshold=4.0, window=60):
        super().__init__(threshold)
        self.window = window

    def score(self, s):
        mu = s.rolling(self.window, min_periods=20).mean().shift(1)
        sd = s.rolling(self.window, min_periods=20).std().shift(1)
        z = (s - mu) / sd
        return z.replace([np.inf, -np.inf], np.nan).fillna(0.0)

    def __repr__(self):
        return f"{self.name}(threshold={self.threshold}, window={self.window})"


class EwmaVolDetector(OutlierDetector):
    """Scale each return by trailing EWMA volatility."""

    def __init__(self, threshold=4.0, halflife=20):
        super().__init__(threshold)
        self.halflife = halflife

    def score(self, s):
        vol = s.ewm(halflife=self.halflife, min_periods=20).std().shift(1)
        z = s / vol
        return z.replace([np.inf, -np.inf], np.nan).fillna(0.0)

    def __repr__(self):
        return f"{self.name}(threshold={self.threshold}, halflife={self.halflife})"


detectors = [ZScoreDetector(4.0), ModifiedZScoreDetector(4.0),
             RollingZScoreDetector(4.0, window=60), EwmaVolDetector(4.0, halflife=20)]

for d in detectors:
    print(" ", d)

  ZScore(threshold=4.0)
  ModifiedZScore(threshold=4.0)
  RollingZScore(threshold=4.0, window=60)
  EwmaVol(threshold=4.0, halflife=20)


In [8]:
# run every detector over every series
flags = []
for col in returns.columns:
    s = returns[col].dropna()
    row = {"series": col, "observations": len(s)}
    for d in detectors:
        row[d.name] = int(d.detect(s).sum())
    flags.append(row)

flag_counts = pd.DataFrame(flags).set_index("series")
print(f"observations flagged at {detectors[0].threshold} sigma\n")
print(flag_counts.to_string())

# where the methods disagree is where the interesting cases are
s = returns["sp500"].dropna()
masks = {d.name: d.detect(s) for d in detectors}
agree_all = np.logical_and.reduce(list(masks.values()))
agree_any = np.logical_or.reduce(list(masks.values()))
print(f"\non sp500: {agree_any.sum()} days flagged by at least one method, "
      f"{agree_all.sum()} by all four")
print("the gap between those two numbers is the disagreement worth reviewing.")

observations flagged at 4.0 sigma

        observations  ZScore  ModifiedZScore  RollingZScore  EwmaVol
series                                                              
sp500           2416      16              63             24       14
eurusd          2561       5              19              9        8
usdjpy          2561      16              52             18       17
wti             2571      16              53             18       13
ust10           2565       7              17              7        4
vix             2658      20              43             21       19

on sp500: 70 days flagged by at least one method, 4 by all four
the gap between those two numbers is the disagreement worth reviewing.


In [9]:
s = returns["sp500"].dropna()
ewma_flag = EwmaVolDetector(4.0).detect(s)
z_flag = ZScoreDetector(4.0).detect(s)

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.09,
                    row_heights=[0.55, 0.45],
                    subplot_titles=("S&P 500 daily log returns with flagged points",
                                    "EWMA volatility scaled score"))

fig.add_trace(go.Scatter(x=s.index, y=s * 100, name="return",
                         line=dict(color=blue, width=1),
                         hovertemplate="%{x|%Y-%m-%d}: %{y:.2f}%<extra></extra>"),
              row=1, col=1)
fig.add_trace(go.Scatter(x=s.index[ewma_flag], y=s[ewma_flag] * 100, mode="markers",
                         name="flagged by EWMA",
                         marker=dict(color=orange, size=8, line=dict(width=1, color="white")),
                         hovertemplate="%{x|%Y-%m-%d}: %{y:.2f}%<extra></extra>"),
              row=1, col=1)

score = EwmaVolDetector(4.0).score(s)
fig.add_trace(go.Scatter(x=score.index, y=score, name="EWMA score",
                         line=dict(color=purple, width=1), showlegend=False),
              row=2, col=1)
for lvl in (4, -4):
    fig.add_hline(y=lvl, line=dict(color=muted, dash="dash", width=1), row=2, col=1)

fig.update_yaxes(title="%", row=1, col=1)
fig.update_yaxes(title="sigma", row=2, col=1)
fig.update_layout(height=640,
                  title="Exhibit 2. Scaling by trailing volatility moves the bar with the market")
fig.show()

print(f"EWMA flagged {int(ewma_flag.sum())} days, static z-score flagged {int(z_flag.sum())}")
print("the two do not flag the same days, which is the subject of the next section.")

EWMA flagged 14 days, static z-score flagged 16
the two do not flag the same days, which is the subject of the next section.


## 4. Detector benchmark

Counting flags does not tell you whether a detector is any good, because the truth is unknown. So the truth gets manufactured: take a clean series, inject outliers at known dates and known sizes, and measure what each method recovers.

**The sizing choice is the whole experiment.** Each injected outlier is sized as a multiple of *local* volatility at that date, not full sample volatility. That is how a real bad print behaves, because a fat finger during a calm week produces a move that is enormous relative to that week but unremarkable against a sample that includes March 2020.

Three metrics:

* **Precision** = of the points flagged, what share were genuinely injected. Low precision means analysts waste time on false alarms.
* **Recall** = of the injected points, what share were caught. Low recall means bad data reaches the risk engine.
* **F1** = the harmonic mean of the two.


In [10]:
class DefectInjector:
    """Injects known defects into a clean series so detectors can be scored."""

    def __init__(self, seed=42):
        self.rng = np.random.default_rng(seed)

    def inject_outliers(self, s, n_outliers=40, size_local_vol=6.0, window=60):
        """Add spikes sized against LOCAL volatility, and return the truth mask."""
        out = s.copy()
        local_vol = s.rolling(window, min_periods=20).std().shift(1).bfill()
        eligible = np.arange(window + 20, len(s))
        idx = self.rng.choice(eligible, min(n_outliers, len(eligible)), replace=False)
        truth = pd.Series(False, index=s.index)
        for i in idx:
            sign = self.rng.choice([-1, 1])
            out.iloc[i] = out.iloc[i] + sign * size_local_vol * local_vol.iloc[i]
            truth.iloc[i] = True
        return out, truth, local_vol.iloc[idx]

    def make_stale(self, prices, n_runs=8, run_length=5):
        """Freeze the price for runs of days, the way a stale vendor feed looks."""
        out = prices.copy()
        starts = self.rng.choice(np.arange(60, len(prices) - run_length - 1),
                                 n_runs, replace=False)
        stale_dates = []
        for s0 in starts:
            for j in range(1, run_length + 1):
                out.iloc[s0 + j] = out.iloc[s0]
                stale_dates.append(prices.index[s0 + j])
        return out, stale_dates

    def add_bad_prints(self, prices, n_bad=6, pct=0.07):
        """Decimal slip style errors: the price is off by a few percent for one day."""
        out = prices.copy()
        idx = self.rng.choice(np.arange(60, len(prices) - 1), n_bad, replace=False)
        bad_dates = []
        for i in idx:
            out.iloc[i] = out.iloc[i] * (1 + self.rng.choice([-1, 1]) * pct)
            bad_dates.append(prices.index[i])
        return out, bad_dates

    def punch_holes(self, prices, n_missing=30):
        """Remove observations at random."""
        out = prices.copy()
        idx = self.rng.choice(np.arange(60, len(prices) - 1), n_missing, replace=False)
        out.iloc[idx] = np.nan
        return out, [prices.index[i] for i in idx]


def score_detection(predicted, truth):
    """Precision, recall and F1 for a boolean prediction against a boolean truth."""
    tp = int((predicted & truth).sum())
    fp = int((predicted & ~truth).sum())
    fn = int((~predicted & truth).sum())
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {"precision": precision, "recall": recall, "f1": f1,
            "true_pos": tp, "false_pos": fp, "false_neg": fn}


injector = DefectInjector(seed=42)
clean_series = returns["sp500"].dropna()
dirty_series, truth_mask, vol_at_injection = injector.inject_outliers(
    clean_series, n_outliers=40, size_local_vol=6.0)

print(f"injected 40 outliers at 6x local volatility")
print(f"  local vol at those dates ranged "
      f"{vol_at_injection.min() * 100:.2f}% to {vol_at_injection.max() * 100:.2f}%")
print(f"  so the same 6x defect is {vol_at_injection.max() / vol_at_injection.min():.1f}x "
      f"larger in absolute terms at one end than the other")

injected 40 outliers at 6x local volatility
  local vol at those dates ranged 0.30% to 1.99%
  so the same 6x defect is 6.6x larger in absolute terms at one end than the other


In [11]:
rows = []
for d in detectors:
    res = score_detection(d.detect(dirty_series), truth_mask)
    res["method"] = d.name
    rows.append(res)

bench = pd.DataFrame(rows).set_index("method")
print("detector benchmark\n")
print(bench.round(3).to_string())

best = bench["f1"].idxmax()
print(f"\nbest F1: {best} at {bench.loc[best, 'f1']:.3f}")

detector benchmark

                precision  recall     f1  true_pos  false_pos  false_neg
method                                                                  
ZScore             0.5380  0.3500 0.4240        14         12         26
ModifiedZScore     0.3570  0.8750 0.5070        35         63          5
RollingZScore      0.6430  0.6750 0.6590        27         15         13
EwmaVol            0.7350  0.6250 0.6760        25          9         15

best F1: EwmaVol at 0.676


In [12]:
# split the injected points by the volatility regime they landed in
median_vol = vol_at_injection.median()
calm_dates = vol_at_injection.index[vol_at_injection <= median_vol]
vol_dates = vol_at_injection.index[vol_at_injection > median_vol]

split = []
for d in detectors:
    pred = d.detect(dirty_series)
    split.append({
        "method": d.name,
        "recall_calm": float(pred.loc[calm_dates].mean()),
        "recall_volatile": float(pred.loc[vol_dates].mean()),
    })

split = pd.DataFrame(split).set_index("method")
split["gap"] = split["recall_volatile"] - split["recall_calm"]
print("recall by volatility regime\n")
print(split.round(3).to_string())

print("\nreading this table:")
print("  a method with a large positive gap only works when markets are already moving.")
print("  it misses defects in calm periods, which is when a bad print is least likely")
print("  to be caught by eye and most likely to reach the risk engine unnoticed.")

recall by volatility regime

                recall_calm  recall_volatile    gap
method                                             
ZScore               0.1000           0.6000 0.5000
ModifiedZScore       0.7500           1.0000 0.2500
RollingZScore        0.6500           0.7000 0.0500
EwmaVol              0.6000           0.6500 0.0500

reading this table:
  a method with a large positive gap only works when markets are already moving.
  it misses defects in calm periods, which is when a bad print is least likely
  to be caught by eye and most likely to reach the risk engine unnoticed.


In [13]:
fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.14,
                    subplot_titles=("Precision, recall and F1",
                                    "Recall by volatility regime"))

for i, metric in enumerate(["precision", "recall", "f1"]):
    fig.add_trace(go.Bar(x=bench.index, y=bench[metric], name=metric,
                         marker_color=palette[i],
                         hovertemplate="%{x}: %{y:.2f}<extra></extra>"),
                  row=1, col=1)

fig.add_trace(go.Bar(x=split.index, y=split["recall_calm"], name="calm",
                     marker_color=blue, showlegend=True,
                     hovertemplate="%{x}: %{y:.2f}<extra></extra>"), row=1, col=2)
fig.add_trace(go.Bar(x=split.index, y=split["recall_volatile"], name="volatile",
                     marker_color=orange, showlegend=True,
                     hovertemplate="%{x}: %{y:.2f}<extra></extra>"), row=1, col=2)

fig.update_yaxes(title="score", range=[0, 1.05], row=1, col=1)
fig.update_yaxes(title="recall", range=[0, 1.05], row=1, col=2)
fig.update_xaxes(tickangle=-25)
fig.update_layout(height=470, barmode="group",
                  title="Exhibit 3. A fixed threshold is not a fixed standard")
fig.show()

### Threshold sensitivity

A detector is only as good as the threshold it is run at, and that threshold is a business decision. Low thresholds catch more defects and generate more false alarms for someone to clear. The sweep below makes the tradeoff explicit rather than leaving it as a magic number in the code.


In [14]:
thresholds = [2.5, 3.0, 3.5, 4.0, 4.5, 5.0, 6.0]
sweep = []
for t in thresholds:
    for cls, kwargs in [(ZScoreDetector, {}), (ModifiedZScoreDetector, {}),
                        (RollingZScoreDetector, {"window": 60}),
                        (EwmaVolDetector, {"halflife": 20})]:
        d = cls(threshold=t, **kwargs)
        res = score_detection(d.detect(dirty_series), truth_mask)
        sweep.append({"threshold": t, "method": d.name, **res})

sweep = pd.DataFrame(sweep)
pivot = sweep.pivot(index="threshold", columns="method", values="f1")
print("F1 by threshold\n")
print(pivot.round(3).to_string())

best_row = sweep.loc[sweep["f1"].idxmax()]
print(f"\nbest overall: {best_row['method']} at threshold {best_row['threshold']}, "
      f"F1 {best_row['f1']:.3f} "
      f"(precision {best_row['precision']:.2f}, recall {best_row['recall']:.2f})")

fig = go.Figure()
for i, m in enumerate(pivot.columns):
    fig.add_trace(go.Scatter(x=pivot.index, y=pivot[m], name=m, mode="lines+markers",
                             line=dict(width=2, color=palette[i]), marker=dict(size=8),
                             hovertemplate="threshold %{x}: F1 %{y:.3f}<extra></extra>"))
fig.update_xaxes(title="threshold (sigma)")
fig.update_yaxes(title="F1", range=[0, 1.05])
fig.update_layout(height=440,
                  title="Exhibit 4. Where each method peaks, and how fast it falls away")
fig.show()

F1 by threshold

method     EwmaVol  ModifiedZScore  RollingZScore  ZScore
threshold                                                
2.5000      0.6550          0.2910         0.5850  0.6000
3.0000      0.7330          0.3630         0.6490  0.6380
3.5000      0.7060          0.4460         0.6600  0.5820
4.0000      0.6760          0.5070         0.6590  0.4240
4.5000      0.6060          0.5790         0.6110  0.3390
5.0000      0.6000          0.6140         0.6360  0.2640
6.0000      0.3920          0.5980         0.4730  0.1630

best overall: EwmaVol at threshold 3.0, F1 0.733 (precision 0.61, recall 0.93)


## 5. Imputation

Once a gap is found, something has to go in it. Three methods, same pattern as before: an abstract base and concrete subclasses.

| Method | How it fills | The catch |
| --- | --- | --- |
| `ForwardFill` | Repeat the last known value | Each method is scored two ways: how close the filled values land to the truth (RMSE), and how much the fill distorts the volatility that VaR is built on. Those two can rank the methods differently, and when they do, the volatility column is the one that matters, because an imputer can sit close to the true prices while still reshaping the return series the risk number is computed from. |
| `LinearInterpolation` | Straight line between the points either side | Uses the value *after* the gap, which is look ahead bias |
| `PeerScaled` | Move the last value by a correlated peer's return | Only as good as the correlation |

The look ahead problem with interpolation deserves emphasis because it looks so harmless. Filling Tuesday from Monday and Wednesday means Tuesday's risk number depends on Wednesday's price. In a backtest that inflates performance; in production it is simply not computable, because Wednesday has not happened yet.

Each method is scored two ways: how close the filled values are to the truth (RMSE), and how much the fill distorts the volatility that VaR is built on. The second matters more, and they do not agree.


In [15]:
class Imputer(ABC):
    """Base class for filling missing observations in a price series."""

    @property
    def name(self):
        return self.__class__.__name__.replace("Imputer", "")

    @abstractmethod
    def impute(self, s):
        ...

    def __repr__(self):
        return f"{self.name}()"


class ForwardFillImputer(Imputer):
    """Carry the last observation forward. Produces zero returns across the gap."""

    def __init__(self, limit=None):
        self.limit = limit

    def impute(self, s):
        return s.ffill(limit=self.limit)


class LinearInterpolationImputer(Imputer):
    """Straight line across the gap. Uses future information."""

    def impute(self, s):
        return s.interpolate(method="linear", limit_direction="both")


class PeerScaledImputer(Imputer):
    """Move the last known value by a correlated peer's return over the same day."""

    def __init__(self, peer, peer_name=""):
        self.peer = peer
        self.peer_name = peer_name

    @property
    def name(self):
        return f"PeerScaled({self.peer_name})" if self.peer_name else "PeerScaled"

    def impute(self, s):
        out = s.copy()
        peer_ret = np.log(self.peer.where(self.peer > 0)).diff()
        for i in np.flatnonzero(out.isna().to_numpy()):
            if i == 0:
                continue
            prev_idx = out.iloc[:i].last_valid_index()
            if prev_idx is None or pd.isna(peer_ret.iloc[i]):
                continue
            out.iloc[i] = out.loc[prev_idx] * np.exp(peer_ret.iloc[i])
        return out.ffill()


# pick the peer by correlation rather than by assumption
# pick the peer by correlation, but only from comparable price series.
# vix and ust10 are excluded on purpose: vix is the most correlated series in
# the panel by absolute value, but it is a volatility index that moves several
# percent a day against roughly one percent for equities, so scaling a price by
# vix returns produces nonsense. correlation alone is not a sufficient test.
target = "sp500"
corrs = returns.corr()[target].drop(target).dropna()

print(f"correlation of {target} with the others:\n")
print(corrs.round(3).to_string())

peer_candidates = [c for c in portfolio_assets if c != target]
eligible = corrs.loc[[c for c in peer_candidates if c in corrs.index]]
peer_name = eligible.abs().idxmax()

print(f"\neligible peers (comparable price series): {', '.join(eligible.index)}")
print(f"strongest eligible peer: {peer_name} (rho = {eligible[peer_name]:.3f})")

excluded = [c for c in corrs.index if c not in eligible.index]
if excluded:
    top_excluded = corrs.loc[excluded].abs().idxmax()
    print(f"\nexcluded {', '.join(excluded)} despite {top_excluded} showing "
          f"rho = {corrs[top_excluded]:.3f}, the largest in the panel.")
    print("a peer has to be comparable in scale, not just correlated.")

correlation of sp500 with the others:

eurusd    0.1160
usdjpy    0.0900
wti       0.1270
ust10     0.1130
vix      -0.7270

eligible peers (comparable price series): eurusd, usdjpy, wti
strongest eligible peer: wti (rho = 0.127)

excluded ust10, vix despite vix showing rho = -0.727, the largest in the panel.
a peer has to be comparable in scale, not just correlated.


In [16]:
def benchmark_imputers(prices, imputers, n_mask=80, seed=7):
    """Mask known values, fill them back, and score accuracy and volatility damage."""
    rng = np.random.default_rng(seed)
    clean = prices.dropna()
    idx = rng.choice(np.arange(60, len(clean) - 1), n_mask, replace=False)
    holed = clean.copy()
    holed.iloc[idx] = np.nan

    true_vol = np.log(clean).diff().std() * np.sqrt(252) * 100
    rows = []
    for im in imputers:
        filled = im.impute(holed)
        err = filled.iloc[idx] - clean.iloc[idx]
        filled_vol = np.log(filled.where(filled > 0)).diff().std() * np.sqrt(252) * 100
        rows.append({
            "method": im.name,
            "rmse": float(np.sqrt((err ** 2).mean())),
            "mape_pct": float((err / clean.iloc[idx]).abs().mean() * 100),
            "ann_vol_pct": float(filled_vol),
            "vol_error_pct": float(filled_vol - true_vol),
        })
    return pd.DataFrame(rows).set_index("method"), true_vol


imputers = [ForwardFillImputer(), LinearInterpolationImputer(),
            PeerScaledImputer(raw[peer_name], peer_name)]

imp_bench, true_vol = benchmark_imputers(raw[target], imputers, n_mask=80)
print(f"imputer benchmark on {target}, 80 masked observations")
print(f"true annualised volatility {true_vol:.2f}%\n")
print(imp_bench.round(3).to_string())

best_rmse = imp_bench["rmse"].idxmin()
best_vol = imp_bench["vol_error_pct"].abs().idxmin()
print(f"\nlowest RMSE:            {best_rmse}")
print(f"least volatility damage: {best_vol}")
if best_rmse != best_vol:
    print("\nthese disagree, and the second one is what matters for VaR.")
    print("an imputer can land close to the true prices while still flattening")
    print("the return series that the risk number is actually computed from.")

imputer benchmark on sp500, 80 masked observations
true annualised volatility 18.12%

                        rmse  mape_pct  ann_vol_pct  vol_error_pct
method                                                            
ForwardFill          63.8080    0.9470      18.4300         0.3100
LinearInterpolation  32.2050    0.5230      17.8300        -0.2900
PeerScaled(wti)     169.9450    2.2840      23.0440         4.9240

lowest RMSE:            LinearInterpolation
least volatility damage: LinearInterpolation


In [17]:
fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.16,
                    subplot_titles=("Accuracy: RMSE of filled values",
                                    "Damage: error in annualised volatility"))

fig.add_trace(go.Bar(x=imp_bench.index, y=imp_bench["rmse"], marker_color=blue,
                     showlegend=False,
                     hovertemplate="%{x}: RMSE %{y:.3f}<extra></extra>"), row=1, col=1)

vol_colors = [orange if v < 0 else green for v in imp_bench["vol_error_pct"]]
fig.add_trace(go.Bar(x=imp_bench.index, y=imp_bench["vol_error_pct"],
                     marker_color=vol_colors, showlegend=False,
                     hovertemplate="%{x}: %{y:+.3f}pp<extra></extra>"), row=1, col=2)
fig.add_hline(y=0, line=dict(color=muted, width=1), row=1, col=2)

fig.update_yaxes(title="RMSE (index points)", row=1, col=1)
fig.update_yaxes(title="volatility error (pp)", row=1, col=2)
fig.update_xaxes(tickangle=-20)
fig.update_layout(height=450,
                  title="Exhibit 5. The most accurate fill is not the safest one")
fig.show()

print("negative volatility error means the fill made the series look calmer")
print("than it really was, which flows straight through to an understated VaR.")

negative volatility error means the fill made the series look calmer
than it really was, which flows straight through to an understated VaR.


## 6. VaR models

Value at Risk answers one question: over a given horizon and confidence level, how much could this book lose on a bad day. A 99% one day VaR of $200,000 means that on 99 days out of 100 the loss should be smaller than that.

Three standard approaches, one base class.

| Model | Assumption | Strength | Weakness |
| --- | --- | --- | --- |
| `HistoricalSimulation` | The future looks like the observed past | No distributional assumption, keeps real fat tails | Every observation is used directly, so one bad print becomes the tail |
| `Parametric` | Returns are normal | Fast and transparent | Normality understates tails; real markets have more extreme days than a normal allows |
| `MonteCarlo` | A specified distribution, sampled | Extends to non linear books and options | Only as good as the assumed distribution |

Historical simulation is the one most exposed to data quality, and that is exactly why this project exists. There is no averaging or smoothing between the input and the answer: the 1st percentile of the return series *is* the VaR.


In [18]:
class VaRModel(ABC):
    """Base class for one day Value at Risk estimators."""

    def __init__(self, confidence=0.99):
        self.confidence = confidence

    @property
    def name(self):
        return self.__class__.__name__.replace("VaR", "")

    @property
    def alpha(self):
        """Tail probability, e.g. 0.01 for 99% confidence."""
        return 1 - self.confidence

    @abstractmethod
    def estimate(self, returns):
        """Return VaR as a positive fraction of portfolio value."""
        ...

    def estimate_dollars(self, returns, portfolio_value):
        return self.estimate(returns) * portfolio_value

    def __repr__(self):
        return f"{self.name}(confidence={self.confidence:.0%})"


class HistoricalSimulationVaR(VaRModel):
    """The empirical quantile of past returns. No distribution assumed."""

    def estimate(self, returns):
        r = returns.dropna()
        if len(r) < 30:
            return np.nan
        return float(-np.percentile(r, self.alpha * 100))


class ParametricVaR(VaRModel):
    """Normal distribution fitted to the sample mean and standard deviation."""

    def estimate(self, returns):
        r = returns.dropna()
        if len(r) < 30:
            return np.nan
        return float(-(r.mean() + stats.norm.ppf(self.alpha) * r.std(ddof=1)))


class MonteCarloVaR(VaRModel):
    """Simulate from a fitted normal, then take the empirical quantile."""

    def __init__(self, confidence=0.99, n_sims=100_000, seed=42):
        super().__init__(confidence)
        self.n_sims = n_sims
        self.seed = seed

    def estimate(self, returns):
        r = returns.dropna()
        if len(r) < 30:
            return np.nan
        rng = np.random.default_rng(self.seed)
        sims = rng.normal(r.mean(), r.std(ddof=1), self.n_sims)
        return float(-np.percentile(sims, self.alpha * 100))

    def __repr__(self):
        return f"{self.name}(confidence={self.confidence:.0%}, n_sims={self.n_sims:,})"


var_models = [HistoricalSimulationVaR(confidence),
              ParametricVaR(confidence),
              MonteCarloVaR(confidence)]

print(f"portfolio value ${portfolio_value:,} at {confidence:.0%} confidence\n")
rows = []
for m in var_models:
    v = m.estimate(portfolio)
    rows.append({"model": m.name, "var_pct": v * 100,
                 "var_dollars": v * portfolio_value})
    print(f"  {m!r}")

var_table = pd.DataFrame(rows).set_index("model")
print()
print(var_table.round(2).to_string())

portfolio value $10,000,000 at 99% confidence

  HistoricalSimulation(confidence=99%)
  Parametric(confidence=99%)
  MonteCarlo(confidence=99%, n_sims=100,000)

                      var_pct  var_dollars
model                                     
HistoricalSimulation   2.2700 226,870.5800
Parametric             2.1100 210,929.6600
MonteCarlo             2.1300 212,516.7400


In [19]:
# how far the empirical tail sits from the normal assumption
r = portfolio.dropna()
excess_kurt = float(stats.kurtosis(r))
skewness = float(stats.skew(r))
jb_stat, jb_p = stats.jarque_bera(r)

print("distribution of portfolio returns\n")
print(f"  observations     {len(r):,}")
print(f"  mean             {r.mean() * 100:+.4f}%")
print(f"  std dev          {r.std() * 100:.4f}%")
print(f"  skewness         {skewness:+.3f}")
print(f"  excess kurtosis  {excess_kurt:+.3f}   (0 for a normal)")
print(f"  jarque bera      {jb_stat:,.1f}  p = {jb_p:.2e}")

if jb_p < 0.05:
    print("\n  normality is rejected. returns have fatter tails than a normal,")
    print("  which is why parametric VaR tends to sit below historical simulation")
    print("  at high confidence levels.")

gap = (var_table.loc["HistoricalSimulation", "var_pct"]
       - var_table.loc["Parametric", "var_pct"])
print(f"\n  historical simulation minus parametric: {gap:+.3f} percentage points "
      f"(${gap / 100 * portfolio_value:+,.0f})")

distribution of portfolio returns

  observations     2,537
  mean             +0.0418%
  std dev          0.9247%
  skewness         +0.191
  excess kurtosis  +24.256   (0 for a normal)
  jarque bera      62,208.1  p = 0.00e+00

  normality is rejected. returns have fatter tails than a normal,
  which is why parametric VaR tends to sit below historical simulation
  at high confidence levels.

  historical simulation minus parametric: +0.159 percentage points ($+15,941)


## 7. What dirty data does to VaR

This is the section the rest of the notebook exists to support.

The experiment: take the clean price panel, inject a realistic mix of defects, recompute the portfolio return series, and recompute VaR. Every defect type is one that appears in real vendor feeds.

| Defect | How it is simulated | What it does to the return series |
| --- | --- | --- |
| Bad prints | Price off by 7% for one day, then corrected | Two large fake returns back to back, one in each direction |
| Staleness | Price frozen for five day runs | Zeros during the freeze, then one catch up move when it resumes |
| Missing values | Observations removed at random | Depends entirely on how they are filled |

Then VaR is computed three ways: on the clean data, on the raw dirty data, and on the dirty data after running it through the detection and imputation pipeline. The third number is what a working quality process would deliver.


In [20]:
# build a dirty copy of the panel with a realistic mix of defects
dirty_prices = raw.copy()
defect_log = {}

for i, col in enumerate(portfolio_assets):
    inj = DefectInjector(seed=100 + i)
    s = raw[col]
    s, bad_dates = inj.add_bad_prints(s, n_bad=6, pct=0.07)
    s, stale_dates = inj.make_stale(s, n_runs=8, run_length=5)
    s, gap_dates = inj.punch_holes(s, n_missing=30)
    dirty_prices[col] = s
    defect_log[col] = {"bad_prints": len(bad_dates), "stale_days": len(stale_dates),
                       "missing": len(gap_dates)}

print("defects injected per series\n")
print(pd.DataFrame(defect_log).T.to_string())

dirty_report = DataQualityReport(dirty_prices[portfolio_assets]).summary()
clean_report = DataQualityReport(raw[portfolio_assets]).summary()

compare = pd.DataFrame({
    "clean_grade": clean_report["grade"],
    "clean_score": clean_report["score"],
    "dirty_grade": dirty_report["grade"],
    "dirty_score": dirty_report["score"],
})
print("\nscorecard picks the damage up\n")
print(compare.to_string())

defects injected per series

        bad_prints  stale_days  missing
sp500            6          40       30
eurusd           6          40       30
usdjpy           6          40       30
wti              6          40       30

scorecard picks the damage up

       clean_grade  clean_score dirty_grade  dirty_score
series                                                  
sp500            B      88.2000           B      80.8000
eurusd           B      89.8000           B      81.5000
usdjpy           B      88.9000           B      81.8000
wti              B      89.4000           B      85.0000


In [21]:
def clean_pipeline(prices, detector, imputer_cls=ForwardFillImputer):
    """Flag outlier returns, null the offending prices, then impute the holes."""
    out = prices.copy()
    for col in out.columns:
        s = out[col]
        rets_col = np.log(s.where(s > 0)).diff()
        flagged = detector.detect(rets_col.dropna())
        bad_dates = flagged.index[flagged]
        s = s.copy()
        s.loc[bad_dates] = np.nan
        out[col] = imputer_cls().impute(s)
    return out


cleaner = EwmaVolDetector(threshold=4.0, halflife=20)
repaired_prices = clean_pipeline(dirty_prices[portfolio_assets], cleaner)

clean_ret = compute_returns(raw[portfolio_assets], return_kind)
dirty_ret = compute_returns(dirty_prices[portfolio_assets], return_kind)
repaired_ret = compute_returns(repaired_prices, return_kind)

clean_port = build_portfolio(clean_ret)
dirty_port = build_portfolio(dirty_ret)
repaired_port = build_portfolio(repaired_ret)

# restrict all three to the same dates, otherwise part of the difference
# would just be sample size rather than data quality
common = clean_port.index.intersection(dirty_port.index).intersection(repaired_port.index)
clean_port = clean_port.loc[common]
dirty_port = dirty_port.loc[common]
repaired_port = repaired_port.loc[common]

print(f"portfolio return series, all on the same {len(common):,} dates\n")
for label, ser in [("clean", clean_port), ("raw (dirty)", dirty_port),
                   ("repaired", repaired_port)]:
    print(f"  {label:<12} n={len(ser):,}  "
          f"ann vol {ser.std() * np.sqrt(252) * 100:6.2f}%  "
          f"worst day {ser.min() * 100:7.2f}%")

portfolio return series, all on the same 2,519 dates

  clean        n=2,519  ann vol  14.69%  worst day   -9.46%
  raw (dirty)  n=2,519  ann vol  15.34%  worst day   -9.46%
  repaired     n=2,519  ann vol  14.26%  worst day   -8.48%


In [22]:
rows = []
for m in var_models:
    c = m.estimate(clean_port)
    d = m.estimate(dirty_port)
    p = m.estimate(repaired_port)
    rows.append({
        "model": m.name,
        "clean_pct": c * 100, "raw_pct": d * 100, "repaired_pct": p * 100,
        "raw_error_pct": (d / c - 1) * 100,
        "repaired_error_pct": (p / c - 1) * 100,
        "raw_error_usd": (d - c) * portfolio_value,
        "repaired_error_usd": (p - c) * portfolio_value,
    })

impact = pd.DataFrame(rows).set_index("model")
print(f"VaR at {confidence:.0%} on a ${portfolio_value:,} portfolio\n")
print(impact.round(3).to_string())

hs_raw_err = impact.loc["HistoricalSimulation", "raw_error_usd"]
hs_rep_err = impact.loc["HistoricalSimulation", "repaired_error_usd"]
recovered = (1 - abs(hs_rep_err) / abs(hs_raw_err)) * 100 if hs_raw_err else np.nan

print(f"\nhistorical simulation, the model most exposed to raw data:")
print(f"  uncleaned, VaR is off by ${hs_raw_err:+,.0f} "
      f"({impact.loc['HistoricalSimulation', 'raw_error_pct']:+.1f}%)")
print(f"  after the pipeline, off by ${hs_rep_err:+,.0f} "
      f"({impact.loc['HistoricalSimulation', 'repaired_error_pct']:+.1f}%)")
if np.isfinite(recovered):
    print(f"  the cleaning step recovers {recovered:.0f}% of the error")

VaR at 99% on a $10,000,000 portfolio

                      clean_pct  raw_pct  repaired_pct  raw_error_pct  repaired_error_pct  raw_error_usd  repaired_error_usd
model                                                                                                                       
HistoricalSimulation     2.2700   2.4060        2.2590         5.9950             -0.4870    13,608.3040         -1,105.2730
Parametric               2.1110   2.2050        2.0520         4.4370             -2.8090     9,367.5440         -5,929.5190
MonteCarlo               2.1270   2.2210        2.0670         4.4370             -2.8090     9,437.6650         -5,975.7010

historical simulation, the model most exposed to raw data:
  uncleaned, VaR is off by $+13,608 (+6.0%)
  after the pipeline, off by $-1,105 (-0.5%)
  the cleaning step recovers 92% of the error


In [23]:
fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.15,
                    subplot_titles=(f"VaR at {confidence:.0%} ($)",
                                    "Error against the clean benchmark ($)"))

for i, (label, col, colour) in enumerate([
        ("clean", "clean_pct", blue), ("raw", "raw_pct", orange),
        ("repaired", "repaired_pct", green)]):
    fig.add_trace(go.Bar(x=impact.index, y=impact[col] / 100 * portfolio_value,
                         name=label, marker_color=colour,
                         hovertemplate="%{x}: $%{y:,.0f}<extra></extra>"),
                  row=1, col=1)

fig.add_trace(go.Bar(x=impact.index, y=impact["raw_error_usd"], name="raw error",
                     marker_color=orange, showlegend=False,
                     hovertemplate="%{x}: $%{y:+,.0f}<extra></extra>"), row=1, col=2)
fig.add_trace(go.Bar(x=impact.index, y=impact["repaired_error_usd"],
                     name="repaired error", marker_color=green, showlegend=False,
                     hovertemplate="%{x}: $%{y:+,.0f}<extra></extra>"), row=1, col=2)
fig.add_hline(y=0, line=dict(color=muted, width=1), row=1, col=2)

fig.update_yaxes(title="VaR ($)", row=1, col=1)
fig.update_yaxes(title="error ($)", row=1, col=2)
fig.update_xaxes(tickangle=-20)
fig.update_layout(height=470, barmode="group",
                  title="Exhibit 6. Data defects move the risk number by real money")
fig.show()

### Isolating each defect

The combined result above mixes effects that pull against each other, so each defect is run on its own below, and against **two** VaR models. Running both is the point: historical simulation is a quantile of the observed returns, while parametric VaR is a function of their variance, so a defect that reshapes the tail and a defect that changes the spread do not register the same way.

Three mechanisms, and they are genuinely different:

* **Bad prints inflate both models.** A 7% error creates a return larger than anything real in the sample, which lands in the tail and lifts the variance at the same time.
* **Staleness dilutes variance but leaves the tail alone.** A frozen feed produces zero returns, which pulls annualised volatility down and takes parametric VaR with it. The empirical 1st percentile is unaffected, because the extreme days in the sample are untouched.
* **Forward filled gaps inflate the tail without touching variance.** This is the one that is easy to get backwards. Filling a gap makes that day a zero return, but the *next* day's return then spans two days. In a sample that contains March 2020, a merged two day return can exceed anything real, so it becomes the tail. The zeros and the doubled moves roughly cancel in variance terms, which is why parametric VaR barely moves.

So the defect that matters most depends on the model. Variance based measures are damaged by dilution; quantile based measures are damaged by merging. A quality process that only monitors one of them will miss the other, which is the argument for grading the data directly rather than inferring its health from whether the risk number looks reasonable.

In [24]:
hs = HistoricalSimulationVaR(confidence)
base_var = hs.estimate(clean_port)

scenarios = {}

# bad prints only
p = raw[portfolio_assets].copy()
for i, col in enumerate(portfolio_assets):
    p[col], _ = DefectInjector(seed=200 + i).add_bad_prints(p[col], n_bad=6, pct=0.07)
scenarios["bad prints only"] = p

# staleness only
p = raw[portfolio_assets].copy()
for i, col in enumerate(portfolio_assets):
    p[col], _ = DefectInjector(seed=300 + i).make_stale(p[col], n_runs=15, run_length=5)
scenarios["staleness only"] = p

# missing values, forward filled
p = raw[portfolio_assets].copy()
for i, col in enumerate(portfolio_assets):
    holed, _ = DefectInjector(seed=400 + i).punch_holes(p[col], n_missing=60)
    p[col] = ForwardFillImputer().impute(holed)
scenarios["gaps, forward filled"] = p

para = ParametricVaR(confidence)
base_para = para.estimate(clean_port)

rows = []
for label, prices in scenarios.items():
    port = build_portfolio(compute_returns(prices, return_kind)).reindex(common).dropna()
    rows.append({
        "scenario": label,
        "hs_var_pct": hs.estimate(port) * 100,
        "hs_change_usd": (hs.estimate(port) - base_var) * portfolio_value,
        "param_var_pct": para.estimate(port) * 100,
        "param_change_usd": (para.estimate(port) - base_para) * portfolio_value,
        "ann_vol_pct": port.std() * np.sqrt(252) * 100,
    })

isolated = pd.DataFrame(rows).set_index("scenario")
clean_vol = clean_port.std() * np.sqrt(252) * 100
print(f"clean HS VaR {base_var * 100:.3f}% (${base_var * portfolio_value:,.0f})")
print(f"clean parametric VaR {base_para * 100:.3f}% (${base_para * portfolio_value:,.0f})")
print(f"clean annualised volatility {clean_vol:.2f}%\n")
print(isolated.round(3).to_string())

def direction(change_usd, tol=1.0):
    """Describe the effect, treating sub dollar moves as no change."""
    if abs(change_usd) < tol:
        return "leaves VaR unchanged"
    return "overstates" if change_usd > 0 else "understates"


print("\ndirection of each effect:\n")
for label, row in isolated.iterrows():
    print(f"  {label}")
    print(f"    historical simulation {direction(row['hs_change_usd'])} "
          f"(${row['hs_change_usd']:+,.0f})")
    print(f"    parametric            {direction(row['param_change_usd'])} "
          f"(${row['param_change_usd']:+,.0f})")
    print(f"    annualised vol {row['ann_vol_pct']:.2f}% against {clean_vol:.2f}% clean")

clean HS VaR 2.270% ($226,990)
clean parametric VaR 2.111% ($211,109)
clean annualised volatility 14.69%

                      hs_var_pct  hs_change_usd  param_var_pct  param_change_usd  ann_vol_pct
scenario                                                                                     
bad prints only           2.4070    13,727.8550         2.1700        5,917.4680      15.0820
staleness only            2.2700         0.0000         2.0670       -4,420.0180      14.3890
gaps, forward filled      2.3870    11,690.4810         2.1210        1,006.4520      14.7440

direction of each effect:

  bad prints only
    historical simulation overstates ($+13,728)
    parametric            overstates ($+5,917)
    annualised vol 15.08% against 14.69% clean
  staleness only
    historical simulation leaves VaR unchanged ($+0)
    parametric            understates ($-4,420)
    annualised vol 14.39% against 14.69% clean
  gaps, forward filled
    historical simulation overstates ($+11,690)


In [25]:
fig = go.Figure()
fig.add_trace(go.Bar(x=isolated.index, y=isolated["hs_change_usd"],
                     name="historical simulation", marker_color=blue,
                     text=[f"${v:+,.0f}" for v in isolated["hs_change_usd"]],
                     textposition="outside",
                     hovertemplate="%{x}: $%{y:+,.0f}<extra></extra>"))
fig.add_trace(go.Bar(x=isolated.index, y=isolated["param_change_usd"],
                     name="parametric", marker_color=orange,
                     text=[f"${v:+,.0f}" for v in isolated["param_change_usd"]],
                     textposition="outside",
                     hovertemplate="%{x}: $%{y:+,.0f}<extra></extra>"))
fig.add_hline(y=0, line=dict(color=muted, width=1.5))
fig.update_yaxes(title="change in VaR ($)")
fig.update_xaxes(tickangle=-12)
fig.update_layout(height=470, barmode="group",
                  title="Exhibit 7. The same defect moves the two models differently")
fig.show()

print("above zero the desk holds capital it does not need.")
print("below zero it holds less than it should, which is the failure that matters.")
print("\nwhere the two bars disagree, the defect is reshaping the tail and the")
print("variance in different directions, and which model you run decides what")
print("you see.")

above zero the desk holds capital it does not need.
below zero it holds less than it should, which is the failure that matters.

where the two bars disagree, the defect is reshaping the tail and the
variance in different directions, and which model you run decides what
you see.


## 8. Backtesting

A VaR model that is never checked is an assumption. Backtesting compares each day's forecast against the loss that actually happened and counts the **breaches**, meaning days where the loss exceeded VaR.

At 99% confidence roughly 1% of days should breach. Too many and the model understates risk. Too few and it is tying up capital unnecessarily.

**The Kupiec proportion of failures test** turns that into a formal hypothesis test. The statistic is

$$LR = -2\ln\left[\frac{(1-p)^{N-x}\,p^{x}}{(1-\hat{p})^{N-x}\,\hat{p}^{x}}\right]$$

where $p$ is the expected breach rate, $x$ the observed breaches, $N$ the observations and $\hat{p}=x/N$. Under the null that the model is correctly calibrated, $LR$ follows a chi squared distribution with one degree of freedom, so the critical value at 95% is 3.841.

**The Basel traffic light** is the supervisory version, defined over 250 trading days at 99%: 0 to 4 breaches is green, 5 to 9 is yellow and invites scrutiny, 10 or more is red and triggers a capital multiplier.


In [26]:
class VaRBacktest:
    """Rolling window VaR backtest with the Kupiec test and Basel zone."""

    basel_zones = [(4, "green"), (9, "yellow"), (np.inf, "red")]

    def __init__(self, model, window=500):
        self.model = model
        self.window = window

    def run(self, returns):
        """Walk forward, estimating VaR from the trailing window only."""
        r = returns.dropna()
        if len(r) <= self.window:
            raise ValueError(f"need more than {self.window} observations, got {len(r)}")
        vals = r.to_numpy()
        recs = []
        for i in range(self.window, len(r)):
            past = pd.Series(vals[i - self.window:i])
            v = self.model.estimate(past)
            recs.append({"date": r.index[i], "var": v, "actual": vals[i],
                         "breach": bool(vals[i] < -v)})
        return pd.DataFrame(recs).set_index("date")

    def kupiec(self, breaches, n_obs):
        """Proportion of failures likelihood ratio test."""
        p = self.model.alpha
        x = int(breaches)
        if x == 0:
            lr = -2 * n_obs * np.log(1 - p)
        else:
            p_hat = x / n_obs
            lr = -2 * ((n_obs - x) * np.log(1 - p) + x * np.log(p)
                       - (n_obs - x) * np.log(1 - p_hat) - x * np.log(p_hat))
        lr = max(float(lr), 0.0)
        return lr, float(1 - stats.chi2.cdf(lr, df=1))

    def basel_zone(self, breaches, n_obs):
        """Scale the breach count to a 250 day year and map to a supervisory zone."""
        scaled = breaches * 250 / n_obs
        for cutoff, zone in self.basel_zones:
            if scaled <= cutoff:
                return zone, scaled
        return "red", scaled

    def summarise(self, results):
        n = len(results)
        b = int(results["breach"].sum())
        lr, pval = self.kupiec(b, n)
        zone, scaled = self.basel_zone(b, n)
        return {
            "model": self.model.name,
            "observations": n,
            "breaches": b,
            "breach_rate_pct": b / n * 100,
            "expected_rate_pct": self.model.alpha * 100,
            "kupiec_lr": lr,
            "kupiec_p": pval,
            "verdict": "reject" if pval < 0.05 else "accept",
            "breaches_per_250d": scaled,
            "basel_zone": zone,
        }


backtests = {}
summaries = []
for m in var_models:
    bt = VaRBacktest(m, window=500)
    res = bt.run(clean_port)
    backtests[m.name] = res
    summaries.append(bt.summarise(res))

bt_summary = pd.DataFrame(summaries).set_index("model")
print("backtest on clean data, 500 day rolling window\n")
print(bt_summary.round(3).to_string())
print("\nverdict is against the null that the model is correctly calibrated.")
print("accept means the observed breach count is consistent with the target rate.")

backtest on clean data, 500 day rolling window

                      observations  breaches  breach_rate_pct  expected_rate_pct  kupiec_lr  kupiec_p verdict  breaches_per_250d basel_zone
model                                                                                                                                      
HistoricalSimulation          2019        31           1.5350             1.0000     5.0240    0.0250  reject             3.8390      green
Parametric                    2019        47           2.3280             1.0000    26.1670    0.0000  reject             5.8200     yellow
MonteCarlo                    2019        46           2.2780             1.0000    24.4720    0.0000  reject             5.6960     yellow

verdict is against the null that the model is correctly calibrated.
accept means the observed breach count is consistent with the target rate.


In [27]:
# the same backtest on uncleaned data
# dirty_port is already aligned to the same dates as clean_port above,
# so the breach counts are directly comparable
dirty_summaries = []
for m in var_models:
    bt = VaRBacktest(m, window=500)
    try:
        res = bt.run(dirty_port)
        dirty_summaries.append(bt.summarise(res))
    except ValueError as e:
        print(f"  {m.name}: {e}")

dirty_bt = pd.DataFrame(dirty_summaries).set_index("model")
print("backtest on raw (dirty) data\n")
print(dirty_bt.round(3).to_string())

side_by_side = pd.DataFrame({
    "clean_breaches": bt_summary["breaches"],
    "clean_rate_pct": bt_summary["breach_rate_pct"],
    "clean_zone": bt_summary["basel_zone"],
    "raw_breaches": dirty_bt["breaches"],
    "raw_rate_pct": dirty_bt["breach_rate_pct"],
    "raw_zone": dirty_bt["basel_zone"],
})
print("\nclean against raw\n")
print(side_by_side.round(3).to_string())

backtest on raw (dirty) data

                      observations  breaches  breach_rate_pct  expected_rate_pct  kupiec_lr  kupiec_p verdict  breaches_per_250d basel_zone
model                                                                                                                                      
HistoricalSimulation          2019        29           1.4360             1.0000     3.4210    0.0640  accept             3.5910      green
Parametric                    2019        46           2.2780             1.0000    24.4720    0.0000  reject             5.6960     yellow
MonteCarlo                    2019        46           2.2780             1.0000    24.4720    0.0000  reject             5.6960     yellow

clean against raw

                      clean_breaches  clean_rate_pct clean_zone  raw_breaches  raw_rate_pct raw_zone
model                                                                                               
HistoricalSimulation              31          1.

In [28]:
res = backtests["HistoricalSimulation"]
breach_days = res[res["breach"]]

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.09,
                    row_heights=[0.62, 0.38],
                    subplot_titles=("Daily return against the VaR threshold",
                                    "Cumulative breaches versus expectation"))

fig.add_trace(go.Scatter(x=res.index, y=res["actual"] * 100, name="daily return",
                         line=dict(color=blue, width=1),
                         hovertemplate="%{x|%Y-%m-%d}: %{y:.2f}%<extra></extra>"),
              row=1, col=1)
fig.add_trace(go.Scatter(x=res.index, y=-res["var"] * 100, name="VaR threshold",
                         line=dict(color=purple, width=2),
                         hovertemplate="%{x|%Y-%m-%d}: %{y:.2f}%<extra></extra>"),
              row=1, col=1)
fig.add_trace(go.Scatter(x=breach_days.index, y=breach_days["actual"] * 100,
                         mode="markers", name="breach",
                         marker=dict(color=orange, size=9,
                                     line=dict(width=1, color="white")),
                         hovertemplate="%{x|%Y-%m-%d}: %{y:.2f}%<extra></extra>"),
              row=1, col=1)

cum = res["breach"].cumsum()
expected = np.arange(1, len(res) + 1) * (1 - confidence)
fig.add_trace(go.Scatter(x=res.index, y=cum, name="actual breaches",
                         line=dict(color=orange, width=2)), row=2, col=1)
fig.add_trace(go.Scatter(x=res.index, y=expected, name="expected",
                         line=dict(color=muted, width=2, dash="dash")), row=2, col=1)

fig.update_yaxes(title="%", row=1, col=1)
fig.update_yaxes(title="count", row=2, col=1)
fig.update_layout(height=680,
                  title="Exhibit 8. Historical simulation backtest, 500 day rolling window")
fig.show()

s = bt_summary.loc["HistoricalSimulation"]
print(f"{int(s['breaches'])} breaches in {int(s['observations']):,} days "
      f"({s['breach_rate_pct']:.2f}%, target {s['expected_rate_pct']:.2f}%)")
print(f"kupiec LR {s['kupiec_lr']:.3f}, p {s['kupiec_p']:.4f}, "
      f"{s['verdict']} the model")
print(f"basel zone: {s['basel_zone']} "
      f"({s['breaches_per_250d']:.1f} breaches per 250 days)")

31 breaches in 2,019 days (1.54%, target 1.00%)
kupiec LR 5.024, p 0.0250, reject the model
basel zone: green (3.8 breaches per 250 days)


### Breach clustering

Kupiec counts breaches but ignores *when* they happen. A model that produces its whole year of breaches in one week is badly calibrated even if the total is right, because it means risk is being underestimated exactly when it matters.

A quick independence check: under a correctly specified model, a breach today should say nothing about a breach tomorrow.


In [29]:
b = res["breach"].astype(int)
joint = pd.crosstab(b.shift(1).dropna().astype(int), b.iloc[1:].astype(int))
print("transition counts, rows are yesterday and columns are today\n")
print(joint.to_string())

n01 = joint.loc[0, 1] if (0 in joint.index and 1 in joint.columns) else 0
n11 = joint.loc[1, 1] if (1 in joint.index and 1 in joint.columns) else 0
n00 = joint.loc[0, 0] if (0 in joint.index and 0 in joint.columns) else 0
n10 = joint.loc[1, 0] if (1 in joint.index and 0 in joint.columns) else 0

pi01 = n01 / (n00 + n01) if (n00 + n01) else 0.0
pi11 = n11 / (n10 + n11) if (n10 + n11) else 0.0

print(f"\nP(breach | no breach yesterday) = {pi01:.4f}")
print(f"P(breach | breach yesterday)    = {pi11:.4f}")

if pi11 > pi01 * 1.5 and (n10 + n11) > 0:
    print("\nbreaches cluster: a breach makes another one more likely the next day.")
    print("that is the signature of volatility clustering that the model is not")
    print("capturing. an EWMA or GARCH weighted VaR would respond faster.")
else:
    print("\nno strong clustering in this sample.")

transition counts, rows are yesterday and columns are today

breach     0   1
breach          
0       1960  27
1         27   4

P(breach | no breach yesterday) = 0.0136
P(breach | breach yesterday)    = 0.1290

breaches cluster: a breach makes another one more likely the next day.
that is the signature of volatility clustering that the model is not
capturing. an EWMA or GARCH weighted VaR would respond faster.


## 9. Vectorisation

The backtest above re estimates VaR once per day inside a Python loop. For historical simulation that is wasteful, because the whole calculation is a rolling quantile and NumPy can do it in one pass with a strided view.

This matters at production scale. A few thousand days on one portfolio is fine in a loop; thousands of instruments re run daily is not.


In [30]:
def rolling_var_loop(returns, window=500, conf=0.99):
    """One quantile per day, computed in a Python loop."""
    vals = returns.dropna().to_numpy()
    out = np.empty(len(vals) - window)
    for i in range(window, len(vals)):
        out[i - window] = -np.percentile(vals[i - window:i], (1 - conf) * 100)
    return out


def rolling_var_vectorised(returns, window=500, conf=0.99):
    """Build every window as a strided view, then one quantile call over an axis."""
    vals = returns.dropna().to_numpy()
    windows = np.lib.stride_tricks.sliding_window_view(vals, window)[:-1]
    return -np.percentile(windows, (1 - conf) * 100, axis=1)


t0 = time.perf_counter()
slow = rolling_var_loop(clean_port)
t_loop = time.perf_counter() - t0

t0 = time.perf_counter()
fast = rolling_var_vectorised(clean_port)
t_vec = time.perf_counter() - t0

assert np.allclose(slow, fast), "the two implementations must agree"

print(f"rolling 500 day historical VaR over {len(slow):,} days\n")
print(f"  python loop   {t_loop * 1000:8.1f} ms")
print(f"  vectorised    {t_vec * 1000:8.1f} ms")
print(f"  speedup       {t_loop / t_vec:8.1f}x")
print(f"\n  results identical to {np.abs(slow - fast).max():.2e}")
print("\n  the assert is the important line. an optimisation that changes the")
print("  answer is not an optimisation, so the fast path is checked against")
print("  the obvious one rather than trusted.")

rolling 500 day historical VaR over 2,019 days

  python loop      139.9 ms
  vectorised         8.4 ms
  speedup           16.6x

  results identical to 0.00e+00

  the assert is the important line. an optimisation that changes the
  answer is not an optimisation, so the fast path is checked against
  the obvious one rather than trusted.


## 10. Limitations and future work

What this project does not do, and what would come next.

### Limitations

**The defects are simulated, not observed.** Real vendor errors are messier: correlated across instruments, clustered around holidays and corporate actions, and sometimes systematically biased rather than random. The injected defects are a reasonable caricature, not a sample from the real distribution.

**Detection runs one series at a time.** A cross sectional check is often stronger than a time series one. If every bank stock moves 2% and one moves 9%, that single name is suspicious in a way no univariate detector can see. A residual based screen against a factor model would catch it.

**Parametric and Monte Carlo VaR both assume normality.** The Jarque Bera test in section 6 rejects that assumption on this data. A Student t or a fitted extreme value tail would be more honest, and Monte Carlo only earns its complexity once the book contains options, since for a linear portfolio it just reproduces the parametric answer.

**No volatility model.** VaR here is unconditional. EWMA or GARCH weighting would respond faster to regime changes, and the breach clustering check in section 8 is the evidence that it would help.

**Portfolio construction is deliberately trivial.** Equal weights, no FX translation, no position level detail. That keeps the focus on data quality rather than on portfolio mechanics, but it means the dollar figures are illustrative.

**Average Daily Trading Volume is absent** because FRED does not publish volume data. The computation itself is a rolling mean; the hard part is the production plumbing, which the pipeline pattern here demonstrates on prices instead.

### Future work

1. Cross sectional and factor residual outlier detection across instruments.
2. EWMA and GARCH weighted VaR, then re run the backtest to see whether clustering disappears.
3. Expected shortfall alongside VaR, since Basel FRTB moved to ES at 97.5%.
4. A vendor comparison layer that scores the same instrument from two sources against each other, which is how a real quality process resolves a disputed print.
5. Package the classes into an installable library with a pytest suite and a scheduled run, so the scorecard becomes a daily artefact rather than a notebook output.

### References

* Kupiec, P. (1995). Techniques for Verifying the Accuracy of Risk Measurement Models. *Journal of Derivatives*.
* Christoffersen, P. (1998). Evaluating Interval Forecasts. *International Economic Review*.
* Basel Committee on Banking Supervision (1996, revised 2019). Supervisory Framework for the Use of Backtesting.
* Iglewicz, B. and Hoaglin, D. (1993). How to Detect and Handle Outliers. ASQC Quality Press.

### Author

**Aby Joe Jose**, MS Applied Data Science, Clarkson University.
[GitHub](https://github.com/AbyJoeJose) | [Portfolio](https://abyjoejose.github.io/Data-Science-Portfolio) | [LinkedIn](https://linkedin.com/in/aby-joe-jose-88959021b) | abyjoejose00@gmail.com

Independent analysis on public data. Not investment advice, and no affiliation with any financial institution.
